# references
**Prerequisites:** memory_model

**Outcome:** After this notebook you will understand exactly what a reference is, how reference counting works, when Python reclaims memory, and how to trace references in your own code.

## The Problem

In [ ]:
import sys

records = ["job_1", "job_2", "job_3"]
backup  = records
cache   = [records, records]   # stored twice in a list

print(sys.getrefcount(records))  # how many things are pointing at this list?
# why is the number higher than you expect?

## The Concept

A reference is a pointer to an object in memory. Every time you assign a variable, pass an argument, store a value in a container, or return from a function, Python creates a new reference to that object — not a new object. Python tracks how many references exist for each object using a reference count. When that count reaches zero the object's memory is immediately reclaimed. Understanding references tells you exactly when memory is held and when it is released.

## Minimal Example

In [ ]:
import sys

x = ["api", "worker"]
print(sys.getrefcount(x))   # 2: one for x, one for getrefcount's argument

y = x
print(sys.getrefcount(x))   # 3: x, y, getrefcount argument

del y
print(sys.getrefcount(x))   # 2: back to x and getrefcount argument

## How Python Does It

CPython stores a reference count inside every object header. Each new reference increments it. Each deletion or rebinding decrements it. When it hits zero, CPython calls the object's deallocator immediately — no waiting for a garbage collection cycle. This is why CPython memory behaviour is predictable in most cases. PyPy and other implementations may behave differently because they use tracing garbage collectors instead.

In [ ]:
import sys

service = {"name": "api", "status": "running"}
print("created         :", sys.getrefcount(service))

registry = [service]
print("added to list   :", sys.getrefcount(service))

alias = service
print("aliased         :", sys.getrefcount(service))

del alias
print("alias deleted   :", sys.getrefcount(service))

registry.clear()
print("removed from list:", sys.getrefcount(service))

## Building Up

In [ ]:
# every container slot is a reference — not a copy
job = {"id": "job_1", "status": "pending"}
queue   = [job]
archive = [job]

job["status"] = "done"

print(queue[0])    # {"id": "job_1", "status": "done"}
print(archive[0])  # {"id": "job_1", "status": "done"}
# both see the change because both hold a reference to the same object

In [ ]:
# del removes the name binding, not necessarily the object
data = ["extract", "transform", "load"]
ref  = data

del data             # removes the name 'data', refcount drops by 1
print(ref)           # object still alive because 'ref' holds it
# print(data)        # NameError: 'data' is not defined

In [ ]:
# function calls create a temporary reference for the duration of the call
import sys

def inspect(obj):
    # inside the function, the parameter is an extra reference
    print("inside function:", sys.getrefcount(obj))

payload = {"records": 500}
print("before call    :", sys.getrefcount(payload))
inspect(payload)
print("after call     :", sys.getrefcount(payload))

In [ ]:
# weakref — a reference that does not increment the count
import weakref

class Worker:
    def __init__(self, name):
        self.name = name

w = Worker("ingestion")
weak = weakref.ref(w)

print(weak())        # <__main__.Worker object> — still alive
del w
print(weak())        # None — object was collected, weak ref returns None

In [ ]:
# reference counting in a real batch scenario
import sys

batch = [{"id": i, "value": i * 10} for i in range(5)]
print("batch refcount:", sys.getrefcount(batch))

def flush(b):
    print("inside flush  :", sys.getrefcount(b))
    return len(b)

count = flush(batch)
print("after flush   :", sys.getrefcount(batch))
print("flushed", count, "records")

## Where It Breaks

In [ ]:
# circular reference — refcount never reaches zero without the cyclic GC
class Node:
    def __init__(self, name):
        self.name = name
        self.next = None

a = Node("producer")
b = Node("consumer")
a.next = b
b.next = a    # cycle: a → b → a

del a
del b
# both names are gone but both objects still hold references to each other
# refcount for each is 1, not 0 — memory not freed by refcounting alone

In [ ]:
# using == on references when you mean identity check
a = ["job_1", "job_2"]
b = ["job_1", "job_2"]

print(a == b)    # True  — same values
print(a is b)    # False — different objects
# confusing these two is one of the most common Python bugs

## The Fix

In [ ]:
# break cycles manually to help the refcount collector
class Node:
    def __init__(self, name):
        self.name = name
        self.next = None

a = Node("producer")
b = Node("consumer")
a.next = b
b.next = a

# break the cycle before deleting
a.next = None
del a
del b
# now refcounts reach zero and memory is freed immediately

In [ ]:
# use is only for singletons: None, True, False
result = None

if result is None:       # correct
    print("no result yet")

if result == None:       # works but not idiomatic — avoid
    print("no result yet")

## In a Real System

In [ ]:
# a service registry that holds weak references to avoid keeping services alive
import weakref

class Service:
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return f"Service({self.name})"

registry = {}

def register(service):
    registry[service.name] = weakref.ref(service)

def lookup(name):
    ref = registry.get(name)
    if ref is None:
        return None
    return ref()   # returns None if service was garbage collected

api = Service("api")
register(api)

print(lookup("api"))    # Service(api)
del api
print(lookup("api"))    # None — service was collected, registry didn't hold it alive

## Performance

Creating a reference is O(1) — just an integer increment. Deleting one is O(1) — a decrement and a zero check. The cost comes from circular references, which require the cyclic garbage collector to scan object graphs. In memory-sensitive systems, breaking cycles explicitly is faster than relying on the cyclic GC. Use `weakref` for caches and registries where you don't want to extend object lifetime.

## Anti-Pattern

In [ ]:
# anti-pattern: storing large objects in a long-lived container and forgetting
# the container keeps the refcount above zero forever

results_cache = []

def run_job(job_id):
    result = {"id": job_id, "data": list(range(100_000))}  # large object
    results_cache.append(result)   # refcount stays above 0 forever
    return result["id"]

for i in range(10):
    run_job(i)

# results_cache holds 10 large objects in memory indefinitely
# fix: clear the cache, use a bounded deque, or use weakrefs
results_cache.clear()
print("cache cleared, memory released")

## Interview Signals

- How does CPython decide when to free an object's memory?
- What is the difference between `del x` and setting `x = None`?
- Why do circular references prevent the reference counter from freeing memory?
- When would you use `weakref` over a normal reference?

## Exercise

In [ ]:
import sys

def reference_audit(obj):
    """
    Returns the reference count of obj as seen from the caller.
    Constraint: the number returned must NOT include the reference
    created by this function's own parameter or by getrefcount.
    Subtract the known overhead to return the true caller-visible count.
    """
    return sys.getrefcount(obj)   # bug: this overcounts — fix it


data = ["job_1", "job_2"]
copy = data

# from the caller's perspective, data has 2 references: 'data' and 'copy'
assert reference_audit(data) == 2, f"expected 2, got {reference_audit(data)}"
print("all assertions passed")